In [2]:
import os
import re
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

sys.path.append("/mnt/lareaulab/reliscu/code")

from parse_gtf import *
from junction2psi import *

pd.set_option('display.max_columns', None)

In [3]:
corr_df = pd.read_csv("data/corrs/hahn_2023_cortex_STAR_counts_TMMF_All_130_outliers_removed_Claude_marker_genes_PC1_ctype_abundance_exon_counts_corr.csv", index_col=0)

In [4]:
corr_df.head()

,Gene,All Neuronal,All Glutamatergic,All GABAergic,CGE Class,MGE Class,L4 IT,L6 CT,L6 IT Car3,Sncg,Pvalb,Sst,Sst Chodl,Astro,Oligo,Endo,VLMC,Peri,L4 L5 IT,Vip Sncg
ENSMUSG00000025902_ProteinCoding_1,Sox17,0.704820,0.712606,0.685048,0.592332,0.618572,0.602368,0.550662,0.571557,0.592124,0.575845,0.487906,0.180897,0.726523,0.544935,0.704356,0.150673,0.115180,0.623887,0.608719
ENSMUSG00000025902_ProteinCoding_2,Sox17,0.706066,0.713623,0.682193,0.585042,0.617292,0.616523,0.549923,0.565232,0.586625,0.576586,0.485358,0.171957,0.727870,0.535255,0.706896,0.148453,0.112848,0.625975,0.624372
ENSMUSG00000025902_ProteinCoding_3,Sox17,0.356319,0.361247,0.318949,0.216859,0.283328,0.143687,0.271799,0.282952,0.314251,0.280414,0.232569,0.088818,0.341162,0.289881,0.304612,0.077093,0.043558,0.336717,0.149327
ENSMUSG00000025902_ProteinCoding_4,Sox17,0.465436,0.470932,0.416033,0.279767,0.347572,0.264590,0.364281,0.305548,0.367779,0.434841,0.248640,0.084684,0.432308,0.374428,0.374598,0.033845,0.005814,0.454376,0.251553
ENSMUSG00000033845_ProteinCoding_1,Mrpl15,0.896632,0.866385,0.941678,0.861636,0.937214,0.551563,0.618362,0.920686,0.910495,0.557476,0.859579,0.541323,0.950891,0.585404,0.832032,0.457440,0.467041,0.711256,0.508402


In [ ]:
# L6 CT is oligo

# Sst Chodl is not well defined (excitatory markers)

# Endo is suss (includes neuronal markers, might be metabolic)

# L6 IT Car3 has some evidence of  brain abnormalities (Apoe, Gas5)

# Sncg is not specific... could be broadly neuronal
# MGE Class is the same ^
# CGE Class is the same ^

# All Glutamatergic looks broadly neuronal
# All GABAergic looks like a housekeeping gene signature

In [32]:
corr_df.sort_values("All GABAergic", ascending=False).head(50)

,Gene,All Neuronal,All Glutamatergic,All GABAergic,CGE Class,MGE Class,L4 IT,L6 CT,L6 IT Car3,Sncg,Pvalb,Sst,Sst Chodl,Astro,Oligo,Endo,VLMC,Peri,L4 L5 IT,Vip Sncg
ENSMUSG00000027523_ProteinCoding_4,Gnas,0.934976,0.908254,0.980993,0.894890,0.962348,0.530051,0.671006,0.905990,0.942193,0.615895,0.849840,0.563172,0.971806,0.662378,0.838399,0.429144,0.445551,0.777189,0.500806
ENSMUSG00000029580_ProteinCoding_1,Actb,0.944052,0.920333,0.980818,0.885345,0.954745,0.581961,0.694524,0.899162,0.926300,0.629081,0.835238,0.522237,0.975977,0.664725,0.855946,0.399881,0.415151,0.784803,0.558849
ENSMUSG00000029580_other_1,Actb,0.933763,0.908936,0.979883,0.894818,0.959059,0.576783,0.678759,0.907840,0.928188,0.612124,0.846485,0.539759,0.972776,0.646964,0.852966,0.414843,0.432612,0.770128,0.549981
ENSMUSG00000027523_ProteinCoding_3,Gnas,0.939425,0.915371,0.979794,0.887222,0.953222,0.536842,0.682901,0.890239,0.930945,0.629943,0.830076,0.546240,0.968656,0.675338,0.834857,0.401735,0.421915,0.788914,0.511676
ENSMUSG00000030695_ProteinCoding_2,Aldoa,0.923449,0.894792,0.974735,0.901317,0.967933,0.513680,0.670222,0.932045,0.954129,0.574666,0.879811,0.575948,0.969424,0.645044,0.838381,0.477543,0.485053,0.748174,0.489304
ENSMUSG00000038965_ProteinCoding_1,Ube2l3,0.907149,0.874616,0.973398,0.907977,0.972054,0.485130,0.634555,0.936228,0.955780,0.560808,0.889551,0.606696,0.959279,0.612461,0.814290,0.497206,0.509908,0.732096,0.444459
ENSMUSG00000022564_ProteinCoding_3,Grina,0.930983,0.906653,0.973383,0.877841,0.951490,0.598576,0.650730,0.905332,0.914202,0.623397,0.839918,0.521311,0.974969,0.622040,0.869590,0.418438,0.429026,0.771265,0.559969
ENSMUSG00000022564_ProteinCoding_2,Grina,0.933160,0.909287,0.972804,0.875653,0.949364,0.598576,0.654860,0.902484,0.912486,0.629260,0.836332,0.517548,0.975101,0.628666,0.869813,0.411000,0.423686,0.775338,0.560352
ENSMUSG00000028843_ProteinCoding_1,Sh3bgrl3,0.951053,0.927298,0.972708,0.871983,0.943327,0.574271,0.705880,0.883428,0.918001,0.644606,0.819885,0.496864,0.976141,0.688551,0.850883,0.371539,0.387978,0.799793,0.572965
ENSMUSG00000022564_ProteinCoding_1,Grina,0.935693,0.912790,0.972637,0.873465,0.947045,0.597902,0.664098,0.897629,0.910407,0.636353,0.831184,0.513392,0.974679,0.639319,0.866431,0.401599,0.416089,0.780292,0.561756


In [22]:
corr_df.loc[corr_df['Gene'].str.contains("Celf4"),]

,Gene,All Neuronal,All Glutamatergic,All GABAergic,CGE Class,MGE Class,L4 IT,L6 CT,L6 IT Car3,Sncg,Pvalb,Sst,Sst Chodl,Astro,Oligo,Endo,VLMC,Peri,L4 L5 IT,Vip Sncg
ENSMUSG00000024268_NMD_1,Celf4,0.726540,0.683662,0.870069,0.890719,0.946049,0.375370,0.409005,0.975215,0.922262,0.267732,0.972456,0.751555,0.845597,0.362318,0.716352,0.694643,0.710424,0.471435,0.313190
ENSMUSG00000024268_ProteinCoding_1,Celf4,0.740486,0.696898,0.876430,0.901441,0.948431,0.328852,0.443044,0.978830,0.939223,0.272856,0.975586,0.764752,0.848897,0.407109,0.704574,0.698455,0.718142,0.487687,0.279900
ENSMUSG00000024268_ProteinCoding_2,Celf4,0.726865,0.682150,0.866423,0.895603,0.942930,0.313726,0.427258,0.978813,0.934438,0.250971,0.978610,0.772423,0.837927,0.390687,0.693671,0.711055,0.730879,0.469239,0.263861


In [6]:
corr_df.iloc[:, 1:].corr()

,All Neuronal,All Glutamatergic,All GABAergic,CGE Class,MGE Class,L4 IT,L6 CT,L6 IT Car3,Sncg,Pvalb,Sst,Sst Chodl,Astro,Oligo,Endo,VLMC,Peri,L4 L5 IT,Vip Sncg
All Neuronal,1.000000,0.996846,0.963520,0.839037,0.897899,0.841191,0.928886,0.804662,0.874436,0.852829,0.721683,0.363468,0.982800,0.910445,0.973343,0.256637,0.240569,0.951905,0.822454
All Glutamatergic,0.996846,1.000000,0.942566,0.799541,0.865492,0.870766,0.946887,0.761375,0.836176,0.884738,0.670851,0.295766,0.968070,0.926911,0.967409,0.185137,0.169266,0.968537,0.857402
All GABAergic,0.963520,0.942566,1.000000,0.951960,0.980988,0.732755,0.809675,0.929718,0.965903,0.689972,0.874639,0.593641,0.993402,0.788751,0.965186,0.491354,0.483264,0.841630,0.688208
CGE Class,0.839037,0.799541,0.951960,1.000000,0.986956,0.529577,0.612080,0.987705,0.985728,0.447450,0.968874,0.801542,0.917176,0.597347,0.864837,0.709696,0.714621,0.648272,0.465559
MGE Class,0.897899,0.865492,0.980988,0.986956,1.000000,0.622620,0.696394,0.980815,0.991324,0.543401,0.949399,0.727609,0.960215,0.671022,0.916550,0.638679,0.634631,0.726461,0.563858
L4 IT,0.841191,0.870766,0.732755,0.529577,0.622620,1.000000,0.840027,0.489721,0.548810,0.883672,0.377713,-0.030756,0.787068,0.789207,0.854508,-0.130088,-0.147128,0.878315,0.987395
L6 CT,0.928886,0.946887,0.809675,0.612080,0.696394,0.840027,1.000000,0.561285,0.674745,0.952806,0.458159,0.053262,0.855033,0.988065,0.853994,-0.038235,-0.068475,0.977464,0.857198
L6 IT Car3,0.804662,0.761375,0.929718,0.987705,0.980815,0.489721,0.561285,1.000000,0.983801,0.383870,0.990011,0.830575,0.896030,0.535740,0.842389,0.765093,0.760933,0.592483,0.420528
Sncg,0.874436,0.836176,0.965903,0.985728,0.991324,0.548810,0.674745,0.983801,1.000000,0.505078,0.961162,0.758671,0.941471,0.657833,0.886276,0.688471,0.676354,0.695447,0.492880
Pvalb,0.852829,0.884738,0.689972,0.447450,0.543401,0.883672,0.952806,0.383870,0.505078,1.000000,0.263484,-0.157956,0.750408,0.947578,0.785019,-0.257566,-0.283234,0.965939,0.906626
